# 02b — Trích xuất 1 file pcap sang CSV, gắn thiết bị theo IP

Khác `02_extract_capture.ipynb` (gắn theo MAC qua `device_map.csv`): ở đây chỉ biết
**IP** của từng thiết bị trong lần bắt gói (Pi làm router, subnet `192.168.10.0/24`).
Điện thoại hay đổi MAC ngẫu nhiên theo mạng, nên IP là khoá tin cậy hơn cho file này.

Quy tắc gắn thiết bị cho mỗi gói:
1. `IP.src` thuộc `IP_TO_DEVICE` → thiết bị đó (gói do thiết bị phát ra).
2. Không có IP khớp (DHCP DISCOVER từ `0.0.0.0`, mDNS qua IPv6...) → tra `Ether.src`
   trong bảng MAC đã **học được** ở lượt 1 (MAC nào gửi gói với IP của thiết bị).
3. DHCP: thêm `client_mac` (`chaddr`) và IP được cấp/xin (`yiaddr`, option 50) để
   cả OFFER/ACK từ Pi cũng quy về đúng thiết bị.

Output (thư mục `Data/features/<tên pcap>/`):
- `packets.csv` — mỗi gói có IP thiết bị ở src **hoặc** dst (cột `direction`).
- `dhcp.csv`, `dns.csv`, `mdns.csv`, `tls.csv` — record thô, cùng cột như `02_extract_capture`.
- `session_features.csv` — 1 dòng / thiết bị, 40 cột qua `aggregate()` của `00_common`,
  kèm nhãn từ `device_labels_capture.csv`.

In [1]:
# Nạp định nghĩa dùng chung từ 00_common.ipynb (không train).
from pathlib import Path

_cwd = Path.cwd().resolve()
_PIPELINE = next((d for p in (_cwd, *_cwd.parents)
                  for d in (p, p / "pipeline", p / "Code" / "pipeline")
                  if (d / "00_common.ipynb").is_file()), None)
assert _PIPELINE is not None, f"Không thấy pipeline/00_common.ipynb quanh {_cwd}"

if not globals().get("SDC_DEFS_LOADED"):
    SDC_IMPORT_ONLY = True
    try:
        get_ipython().run_line_magic("run", f'-i "{_PIPELINE / "00_common.ipynb"}"')
    finally:
        del SDC_IMPORT_ONLY

import pandas as pd
from IPython.display import display

try:
    import scapy
    print("scapy  :", scapy.VERSION)
except ImportError:
    raise SystemExit("Cần scapy để đọc pcap: pip install scapy")

Configured for 88-core CPU
Gốc dự án: D:\01.AI_Security\02.SDC


Đã nạp hàm SDC từ 00_common.ipynb

scapy  :

2.7.0

## 1. Cấu hình

`IP_TO_DEVICE` dùng đúng hostname trong `device_labels_capture.csv` để mục 5 gắn được
nhãn make/type/model. Đổi file pcap hay thêm thiết bị: chỉ sửa ô này.

In [2]:
PCAP_PATH = CAPTURE_DIR / "test_pcap_collection.pcap"

IP_TO_DEVICE = {
    "192.168.10.132": "Samsung-cua-Duc-Anh",
    "192.168.10.118": "Redmi-Note-10",
    "192.168.10.162": "iPhone",
}

OUT_DIR = FEAT_DIR / PCAP_PATH.stem
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert PCAP_PATH.exists(), f"Không tìm thấy {PCAP_PATH}"
print(f"{PCAP_PATH.name}  ({PCAP_PATH.stat().st_size / 1e6:.1f} MB)")
print("Output ->", OUT_DIR)

test_pcap_collection.pcap  (11.5 MB)

Output ->

D:\01.AI_Security\02.SDC\Data\features\test_pcap_collection

## 2. Lượt 1 — học MAC của từng thiết bị từ IP

MAC nào gửi gói có `IP.src` (hoặc ARP `psrc`) là IP thiết bị thì coi là MAC của thiết
bị đó. Cảnh báo nếu một IP có nhiều MAC (đổi MAC giữa chừng) hoặc không thấy gói nào.

In [3]:
from collections import Counter, defaultdict

from scapy.all import ARP, Ether, IP, IPv6, PcapReader, TCP, UDP

ip_macs = defaultdict(Counter)  # ip -> Counter(mac)

with PcapReader(str(PCAP_PATH)) as reader:
    for pkt in reader:
        if not pkt.haslayer(Ether):
            continue
        if pkt.haslayer(IP):
            src_ip = pkt[IP].src
        elif pkt.haslayer(ARP):
            src_ip = pkt[ARP].psrc
        else:
            continue
        if src_ip in IP_TO_DEVICE:
            ip_macs[src_ip][packet_source_mac(pkt)] += 1

mac_to_device = {}
for ip, device in IP_TO_DEVICE.items():
    macs = ip_macs.get(ip)
    if not macs:
        print(f"CẢNH BÁO: không thấy gói nào từ {ip} ({device})")
        continue
    if len(macs) > 1:
        print(f"CẢNH BÁO: {ip} ({device}) có nhiều MAC: {dict(macs)}")
    for mac in macs:
        mac_to_device[mac] = device

display(pd.DataFrame(
    [{"ip": ip, "device": IP_TO_DEVICE[ip], "mac": mac, "n_packets": n}
     for ip, macs in ip_macs.items() for mac, n in macs.items()]
))

,ip,device,mac,n_packets
0,192.168.10.162,iPhone,6e:18:8d:88:08:15,4458
1,192.168.10.132,Samsung-cua-Duc-Anh,7e:f9:e6:c0:a1:aa,231
2,192.168.10.118,Redmi-Note-10,e2:1c:d2:b0:af:a5,2300


## 3. Lượt 2 — trích xuất

Giữ `records_by_device` (record thô) song song với bảng phẳng để mục 5 gọi `aggregate()`
trực tiếp, giống `02_extract_capture`.

In [4]:
from scapy.all import BOOTP, DHCP

from tqdm.auto import tqdm


def device_for_packet(pkt):
    src_ip, _ = _network_addresses(pkt)
    if src_ip in IP_TO_DEVICE:
        return IP_TO_DEVICE[src_ip]
    return mac_to_device.get(packet_source_mac(pkt))


def device_for_dhcp(pkt, dhcp_row):
    device = mac_to_device.get(dhcp_row.get("client_mac"))
    if device:
        return device
    offered = str(pkt[BOOTP].yiaddr) if pkt.haslayer(BOOTP) else None
    requested = _dhcp_options_dict(pkt[DHCP]).get("requested_addr")
    for ip in (offered, requested):
        if ip in IP_TO_DEVICE:
            return IP_TO_DEVICE[ip]
    return device_for_packet(pkt)


def packet_row(pkt, ts):
    src_ip, dst_ip = _network_addresses(pkt)
    if src_ip in IP_TO_DEVICE:
        device, direction = IP_TO_DEVICE[src_ip], "out"
    elif dst_ip in IP_TO_DEVICE:
        device, direction = IP_TO_DEVICE[dst_ip], "in"
    else:
        return None
    l4 = pkt[TCP] if pkt.haslayer(TCP) else pkt[UDP] if pkt.haslayer(UDP) else None
    return {
        "ts": ts,
        "device": device,
        "direction": direction,
        "src_mac": packet_source_mac(pkt),
        "dst_mac": normalize_mac(pkt[Ether].dst),
        "src_ip": src_ip,
        "dst_ip": dst_ip,
        "l4": l4.name if l4 is not None else None,
        "sport": int(l4.sport) if l4 is not None else None,
        "dport": int(l4.dport) if l4 is not None else None,
        "length": len(pkt),
        "top_layer": pkt.lastlayer().name,
    }


packet_rows, dhcp_rows, dns_rows, mdns_rows, tls_rows = [], [], [], [], []
records_by_device = defaultdict(list)
tls_streams = {}  # device -> TLSStreamReassembler

with PcapReader(str(PCAP_PATH)) as reader:
    for pkt in tqdm(reader, desc=PCAP_PATH.name):
        if not pkt.haslayer(Ether):
            continue
        ts = float(pkt.time)

        prow = packet_row(pkt, ts)
        if prow:
            packet_rows.append(prow)

        dhcp_row = extract_dhcp(pkt, ts)
        if dhcp_row:
            device = device_for_dhcp(pkt, dhcp_row)
            if device:
                dhcp_rows.append({"device": device, **dhcp_row})
                records_by_device[device].append(dhcp_row)

        device = device_for_packet(pkt)
        if device is None:
            continue
        src_ip, _ = _network_addresses(pkt)

        for dns_row in extract_dns(pkt, ts):
            row = {"device": device, "src_ip": src_ip, **dns_row}
            (mdns_rows if dns_row["is_mdns"] else dns_rows).append(row)
            records_by_device[device].append(dns_row)

        tls_stream = tls_streams.setdefault(device, TLSStreamReassembler())
        for tls_row in tls_stream.feed(pkt, ts):
            tls_rows.append({"device": device, "src_ip": src_ip, **tls_row})
            records_by_device[device].append(tls_row)

packets_df = pd.DataFrame(packet_rows)
dhcp_df = pd.DataFrame(dhcp_rows)
dns_df = pd.DataFrame(dns_rows)
mdns_df = pd.DataFrame(mdns_rows)
tls_df = pd.DataFrame(tls_rows)

print("packets:", packets_df.shape, "dhcp:", dhcp_df.shape, "dns:", dns_df.shape,
      "mdns:", mdns_df.shape, "tls:", tls_df.shape)

C:\Users\admin\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


test_pcap_collection.pcap: 0it [00:00, ?it/s]

test_pcap_collection.pcap: 355it [00:00, 3515.32it/s]

test_pcap_collection.pcap: 707it [00:00, 3476.26it/s]

test_pcap_collection.pcap: 1055it [00:00, 3391.30it/s]

...

(102, 11)

mdns:

(83, 11)

tls:

(53, 13)

Số record mỗi thiết bị. `dhcp = 0` nghĩa là thiết bị không xin lại IP trong lúc
bắt gói — muốn có DHCP thì bật capture **trước**, rồi tắt/bật Wi-Fi trên điện thoại.

In [5]:
counts = pd.DataFrame({
    "packets": packets_df.groupby("device").size() if not packets_df.empty else None,
    "dhcp": dhcp_df.groupby("device").size() if not dhcp_df.empty else None,
    "dns": dns_df.groupby("device").size() if not dns_df.empty else None,
    "mdns": mdns_df.groupby("device").size() if not mdns_df.empty else None,
    "tls": tls_df.groupby("device").size() if not tls_df.empty else None,
}).reindex(sorted(set(IP_TO_DEVICE.values()))).fillna(0).astype(int)
display(counts)

,packets,dhcp,dns,mdns,tls
device,,,,,
Redmi-Note-10,5676,1,22,33,29
Samsung-cua-Duc-Anh,400,2,21,4,5
iPhone,10177,2,59,46,19


## 4. Lưu CSV thô

Ghi đè mỗi lần chạy.

In [6]:
raw_outputs = {
    "packets.csv": packets_df,
    "dhcp.csv": dhcp_df,
    "dns.csv": dns_df,
    "mdns.csv": mdns_df,
    "tls.csv": tls_df,
}
for name, df in raw_outputs.items():
    df.to_csv(OUT_DIR / name, index=False)
    print(f" - {name:<12} {len(df):>7} dòng")

 - packets.csv    16253 dòng

 - dhcp.csv           5 dòng

 - dns.csv          102 dòng

 - mdns.csv          83 dòng

 - tls.csv           53 dòng

## 5. Gộp thành bảng 40 cột (sẵn sàng cho model)

1 session = 1 thiết bị trong file pcap này. Gọi đúng `aggregate()` của `00_common` trên
record thô — không tự gộp tay, tránh lệch với lúc train/serve.

In [7]:
import csv

frame, feature_groups = load_sessions(SESSIONS_PATH)
features = feature_groups["all"]

with open(CAPTURE_DIR / "device_labels_capture.csv", newline="", encoding="utf-8-sig") as f:
    labels = {row["hostname"].strip(): row for row in csv.DictReader(f)}

ip_of = {device: ip for ip, device in IP_TO_DEVICE.items()}
macs_of = defaultdict(list)
for mac, device in mac_to_device.items():
    macs_of[device].append(mac)

session_rows = []
for device, records in records_by_device.items():
    label = labels.get(device, {})
    session_rows.append({
        "session_id": f"{label.get('canonical_device', device)}_{PCAP_PATH.stem}",
        "device": device,
        "ip": ip_of.get(device),
        "mac": " ".join(macs_of[device]),
        "capture_file": PCAP_PATH.name,
        **aggregate(records, features),
        "canonical_device": label.get("canonical_device"),
        "make": label.get("make"),
        "type": label.get("type"),
        "model": label.get("model"),
    })

session_df = pd.DataFrame(session_rows).sort_values("device")
session_df.to_csv(OUT_DIR / "session_features.csv", index=False)

print(f"{len(session_df)} session x {len(features)} cột feature -> {OUT_DIR / 'session_features.csv'}")
display(session_df[["device", "ip", "mac", "has_dhcp", "has_dns", "has_mdns", "has_tls",
                    "n_sources", "dhcp_vci", "tls_sni_tokens"]])

3 session x 40 cột feature -> D:\01.AI_Security\02.SDC\Data\features\test_pcap_collection\session_features.csv

,device,ip,mac,has_dhcp,has_dns,has_mdns,has_tls,n_sources,dhcp_vci,tls_sni_tokens
1,Redmi-Note-10,192.168.10.118,e2:1c:d2:b0:af:a5,1,1,1,1,4,android-dhcp-12,www google com webcast22 normal c alisg tiktok...
0,Samsung-cua-Duc-Anh,192.168.10.132,7e:f9:e6:c0:a1:aa,1,1,1,1,4,android-dhcp-16,www google com mtalk api samsungcloud z m gate...
2,iPhone,192.168.10.162,6e:18:8d:88:08:15,1,1,1,1,4,<missing>,gdmf apple com captive dns google gsp85 ssl ls...
